# LightGCN (Biased Evaluation) — KuaiRec

Baseline LightGCN on the **biased KuaiRec logs** (`kuairec_combined.csv`) with a user-wise 80/20 split.

- Dataset: `../../data/kuairec_data/kuairec_combined.csv`
- Evaluation: RMSE/MAE on biased test split
- Output: `biased_results_kuairec_lightgcn.csv` in this folder.


In [1]:
import os
import numpy as np
import pandas as pd

import tensorflow as tf
from tensorflow.keras.layers import Input, Embedding, Lambda
from tensorflow.keras.models import Model

from sklearn.metrics import mean_squared_error, mean_absolute_error

print("TensorFlow version:", tf.__version__)


def save_biased_results_csv(
    csv_path,
    dataset_name,
    model_arch,
    estimator_name,
    y_true,
    y_pred,
    ndcg5=None,
    ndcg10=None,
    recall5=None,
    recall10=None,
):
    mse = float(mean_squared_error(y_true, y_pred))
    rmse = float(np.sqrt(mse))
    mae = float(mean_absolute_error(y_true, y_pred))

    row = {
        "Dataset": dataset_name,
        "Architecture": model_arch,
        "Model": estimator_name,
        "MSE": mse,
        "RMSE": rmse,
        "MAE": mae,
        "NDCG@5": np.nan if ndcg5 is None else float(ndcg5),
        "NDCG@10": np.nan if ndcg10 is None else float(ndcg10),
        "Recall@5": np.nan if recall5 is None else float(recall5),
        "Recall@10": np.nan if recall10 is None else float(recall10),
    }

    if os.path.exists(csv_path):
        df = pd.read_csv(csv_path)
        df = pd.concat([df, pd.DataFrame([row])], ignore_index=True)
    else:
        df = pd.DataFrame([row])

    df.to_csv(csv_path, index=False)
    print(f"Saved biased results to: {csv_path}")


TensorFlow version: 2.20.0


In [2]:
# =========
# Load biased KuaiRec logs and create user-wise 80/20 split
# =========

ratings = pd.read_csv('../../data/kuairec_data/kuairec_combined.csv')
if 'user_id' in ratings.columns:
    ratings = ratings.rename(columns={'user_id': 'userId', 'video_id': 'itemId'})
if 'watch_ratio' in ratings.columns and 'rating' not in ratings.columns:
    ratings['rating'] = ratings['watch_ratio']
ratings = ratings[['userId', 'itemId', 'rating']]

train_rows, test_rows = [], []

for user_id, user_data in ratings.groupby('userId'):
    user_data = user_data.sample(frac=1, random_state=42)
    n_items = len(user_data)
    train_size = max(1, int(0.8 * n_items))
    train_rows.append(user_data.iloc[:train_size])
    if train_size < n_items:
        test_rows.append(user_data.iloc[train_size:])

train_df = pd.concat(train_rows)
test_df = pd.concat(test_rows) if test_rows else train_df.sample(frac=0.2, random_state=42)

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

user_ids = pd.concat([train_df['userId'], test_df['userId']]).unique()
item_ids = pd.concat([train_df['itemId'], test_df['itemId']]).unique()

user2idx = {u: i for i, u in enumerate(user_ids)}
item2idx = {i: j for j, i in enumerate(item_ids)}

train_df['user'] = train_df['userId'].map(user2idx)
train_df['item'] = train_df['itemId'].map(item2idx)

test_df['user'] = test_df['userId'].map(user2idx)
test_df['item'] = test_df['itemId'].map(item2idx)

num_users = len(user2idx)
num_items = len(item2idx)

X_train = [train_df['user'].values, train_df['item'].values]
X_test = [test_df['user'].values, test_df['item'].values]

y_train = train_df['rating'].values.astype('float32')
y_test = test_df['rating'].values.astype('float32')

print("Encoded users:", num_users, "items:", num_items)


Train shape: (10021757, 3)
Test shape: (2509049, 3)
Encoded users: 7176 items: 10728


In [3]:
# =========
# LightGCN-style baseline model for KuaiRec
# =========

embedding_dim = 32

user_input = Input(shape=(1,), dtype='int32', name='user_input')
item_input = Input(shape=(1,), dtype='int32', name='item_input')

user_embedding = Embedding(num_users, embedding_dim, name='user_embedding')(user_input)
item_embedding = Embedding(num_items, embedding_dim, name='item_embedding')(item_input)

user_vec = Lambda(lambda x: tf.nn.l2_normalize(x, axis=-1))(user_embedding)
item_vec = Lambda(lambda x: tf.nn.l2_normalize(x, axis=-1))(item_embedding)

score = Lambda(lambda x: tf.reduce_sum(x[0] * x[1], axis=-1, keepdims=True))([user_vec, item_vec])

model = Model(inputs=[user_input, item_input], outputs=score)
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3), loss='mse', metrics=['mae'])

model.summary()


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ user_input          │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ item_input          │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ user_embedding      │ (None, 1, 32)     │    229,632 │ user_input[0][0]  │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ item_embedding      │ (None, 1, 32)     │    343,296 │ item_input[0][0]  │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lambda (Lambda)     │ (None, 1, 32)     │          0 │ user_embedding[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lambda_1 (Lambda)   │ (None, 1, 32)     │          0 │ item_embedding[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lambda_2 (Lambda)   │ (None, 1, 1)      │          0 │ lambda[0][0],     │
│                     │                   │            │ lambda_1[0][0]    │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 572,928 (2.19 MB)

 Trainable params: 572,928 (2.19 MB)

 Non-trainable params: 0 (0.00 B)

In [4]:
# =========
# Train baseline LightGCN on biased KuaiRec split
# =========

callbacks = [
    tf.keras.callbacks.EarlyStopping(patience=2, restore_best_weights=True),
    tf.keras.callbacks.ModelCheckpoint("C:/Users/prana/OneDrive/Documents/thesis/pranathi3/biased data/kuairec/lightgcn_kuairec_best_model.keras", save_best_only=True),
]

history = model.fit(
    X_train,
    y_train,
    validation_data=(X_test, y_test),
    epochs=20,
    batch_size=8192,
    callbacks=callbacks,
    verbose=1,
)


Epoch 1/20
1224/1224 ━━━━━━━━━━━━━━━━━━━━ 561s 457ms/step - loss: 2.8636 - mae: 0.6670 - val_loss: 2.9638 - val_mae: 0.6428
Epoch 2/20
1224/1224 ━━━━━━━━━━━━━━━━━━━━ 543s 443ms/step - loss: 2.7645 - mae: 0.6395 - val_loss: 2.9637 - val_mae: 0.6419
Epoch 3/20
1224/1224 ━━━━━━━━━━━━━━━━━━━━ 576s 470ms/step - loss: 2.7645 - mae: 0.6393 - val_loss: 2.9637 - val_mae: 0.6433
Epoch 4/20
1224/1224 ━━━━━━━━━━━━━━━━━━━━ 526s 429ms/step - loss: 2.7644 - mae: 0.6392 - val_loss: 2.9637 - val_mae: 0.6439
Epoch 5/20
1224/1224 ━━━━━━━━━━━━━━━━━━━━ 529s 432ms/step - loss: 2.7645 - mae: 0.6392 - val_loss: 2.9637 - val_mae: 0.6421
Epoch 6/20
1224/1224 ━━━━━━━━━━━━━━━━━━━━ 532s 434ms/step - loss: 2.7645 - mae: 0.6393 - val_loss: 2.9637 - val_mae: 0.6404
Epoch 7/20
1224/1224 ━━━━━━━━━━━━━━━━━━━━ 541s 442ms/step - loss: 2.7645 - mae: 0.6393 - val_loss: 2.9637 - val_mae: 0.6440


In [5]:
def compute_and_save_biased_results_lightgcn_kuairec():
    """Compute Naive ERM metrics on biased KuaiRec LightGCN split and save to CSV."""
    y_pred = model.predict(X_test, verbose=0).flatten()
    csv_path = "biased_results_kuairec_lightgcn.csv"
    return save_biased_results_csv(
        csv_path=csv_path,
        dataset_name="KUAIREC",
        model_arch="LightGCN",
        estimator_name="Naive ERM",
        y_true=y_test,
        y_pred=y_pred,
    )

print("Call compute_and_save_biased_results_lightgcn_kuairec() after training to write CSV.")


Call compute_and_save_biased_results_lightgcn_kuairec() after training to write CSV.
